# Rotation representations

The IMU measures along its own axes, and an arrow printed on the top side of the flight
  controller marks its forward direction. What generally matters, though, is the Duckiedrone's
  attitude (i.e., how it is turned) in the room, rather than the IMU itself.

The attitude of the Duckiedrone is defined by introducing two reference frames and the rotation
  between them.

The _body frame_ is fixed to the Duckiedrone, centered at its center of mass, and turns and
  moves along with it. To simplify the problem, the IMU axes are **assumed** to coincide with
  the Duckiedrone's body axes (called the "good assembly" assumption below). In fact, on an
  assembled Duckiedrone the flight controller is positioned to be aligned with the Duckiedrone
  "front", as well as being more or less at its center of mass.

The _world frame_ is instead fixed to the room.

The orientation, or attitude, of the Duckiedrone is therefore the rotation that expresses the
  body frame in the world frame.

There are many ways to represent this rotation, and this notebook is about identifying the most
  popular ways to write it down. Note that these are all **different notations for the same
  physical entity**.

Robotics commonly uses three notations:

- Euler angles (roll, pitch and yaw)

- the rotation matrix, also called the direction cosine matrix

- the quaternion.

Each notation has advantages and disadvantages, because despite them being only descriptive
  tools for the same reality, in other words just different points of view of the same entity,
  the tooling itself introduces (mathematical) quirks that require consideration. In particular,
  these different notations differ from a point of view of:

1. Interpretability (from a human perspective): how easy is it to look at the numbers and
   understand what they mean? This might not be relevant for machines, but it definitely is for
   humans.

2. Computational efficiency: how many operations are required to get to the same conclusion,
   e.g., execute a rotation? The Duckiedrone is a fast-moving system (it has a "high
   bandwidth"), so computational time matters, as small delays can destabilize its flight.

3. (Mathematical) Singularities: occasionally math stops making sense, for example when trying
   to divide a number by zero. Different mathematical notations might fail at being meaningful
   in certain "configurations". When a singularity appears in the mathematics of a physical
   process, it is the notation that needs to change, not the physical motion.

The point of this notebook is to clarify which notations will be used going forward, and why, to
  describe rotations.

Most figures in this notebook use an airplane instead of the Duckiedrone because its orientation
  is easier to see. The math is the same for the Duckiedrone.

## Intended Learning Outcomes

By the end of the notebook you should be able to answer:

- What are the three common ways of writing a rotation, and what is each one good and bad at?

- What is gimbal lock, and which motion does it take away?

- Why does the ROS 2 IMU message carry a quaternion rather than roll, pitch and yaw?

- How is a quaternion converted into roll, pitch and yaw, and where does gimbal lock come back?

## Euler angles and the rotation matrix

Both the body and world frames are Cartesian right-handed frames:

- the $x$ axis points "forward",
- the $z$ axis points "up" (opposite the effect of gravity);
- the $y$ axis points "left", in such a way to create a
  [right-handed reference frame](https://en.wikipedia.org/wiki/Right-hand_rule).

Given that only the orientation matters in this notebook (and not translations), both these
  frames are drawn with the same origin, and before any rotation they line up (i.e., when the
  attitude of the Duckiedrone is zero, however it is measured, the Duckiedrone is aligned
  perfectly with the world frame).

One way to describe the orientation is by using three angles, each turning about one axis:

- roll $\alpha$, about $x$, the nose axis
- pitch $\beta$, about $y$, the wing axis
- yaw $\gamma$, about $z$, the vertical axis.

They are often referred to as Euler angles.

<figure style="width:500px; margin:1.5em auto; text-align:center;">
  <img src="../assets/_images/roll-pitch-yaw.jpg" style="width:100%;" />
  <figcaption style="font-size:0.9em; margin-top:0.6em; text-align:center;">
    Roll turns about the nose axis, pitch about the wing axis, and yaw about the vertical axis.
  </figcaption>
</figure>

### One rotation at a time

A turn by the yaw angle $\gamma$ moves a point $\mathbf{p} = [x, y, z]^{T}$ to
  $\mathbf{p}' = R_z(\gamma)\,\mathbf{p}$, where:

$$
R_z(\gamma) =
\begin{bmatrix}
\cos\gamma & -\sin\gamma & 0 \\
\sin\gamma & \cos\gamma & 0 \\
0 & 0 & 1
\end{bmatrix}.
$$

The $z$ coordinate is unchanged, because yaw turns about the $z$ axis, while $x$ and $y$ turn as
  a point on a circle does. Take a Duckiedrone yawed by 90 deg, so $\cos\gamma = 0$ and
  $\sin\gamma = 1$. Its nose, before the rotation, pointed along $[1, 0, 0]^{T}$, and after the
  rotation it points along:

$$
R_z(90^{\circ}) \begin{bmatrix} 1 \\ 0 \\ 0 \end{bmatrix} =
\begin{bmatrix} 0 \\ 1 \\ 0 \end{bmatrix},
$$

the world $y$ axis: a quarter turn to the left. Roll and pitch rotations can be described by
  rotation matrices of the same structure, each leaving its own axis unchanged:

$$
R_y(\beta) =
\begin{bmatrix}
\cos\beta & 0 & \sin\beta \\
0 & 1 & 0 \\
-\sin\beta & 0 & \cos\beta
\end{bmatrix},
\qquad
R_x(\alpha) =
\begin{bmatrix}
1 & 0 & 0 \\
0 & \cos\alpha & -\sin\alpha \\
0 & \sin\alpha & \cos\alpha
\end{bmatrix}.
$$

### Putting the three together

Any arbitrary rotation in 3D space can be decomposed into three subsequent rotations:

$$
R = R_z(\gamma)\,R_y(\beta)\,R_x(\alpha).
$$

Intuitively, the order of the rotations matters, and even mathematically it is made evident by
  the fact that matrix multiplication, which represents rotation operators, does not generally
  commute. Rolling by 90 deg and then pitching by 90 deg, both about the fixed world axes,
  leaves the nose pointing down; pitching first and then rolling leaves it pointing left. So the
  order is part of the convention and has to be stated.

This one is called ZYX. Read right to left, it is roll, then pitch, then yaw, each about the
  fixed world axes. Read left to right, it is yaw, then pitch, then roll, each about the body's
  own turned axes. Both readings give the same $R$. There are 12 different orders of
  [Euler angles](https://en.wikipedia.org/wiki/Euler_angles) that can be used to describe a
  general rotation in 3D space, and this notebook uses only ZYX.

### The rotation matrix is a representation too

$R$ is not only a step on the way to Euler angles: its nine numbers are themselves a way to
  write an orientation, and they have a direct meaning. The first column of $R$ is
  $R\,[1, 0, 0]^{T}$, the direction the body $x$ axis points in the world, and the second and
  third columns are the body $y$ and $z$ axes. For the yawed Duckiedrone above, the first column
  of $R_z(90^{\circ})$ is $[0, 1, 0]^{T}$, the nose direction just computed.

This closes the gap from the start of the notebook. The IMU reports a vector, under the good
  assembly assumption, in the body frame, and one multiplication writes it in the world frame:
  $\mathbf{v}_{\text{world}} = R\,\mathbf{v}_{\text{body}}$. The price is nine numbers for
  something three angles already describe, and each column must keep length 1 and stay at right
  angles to the others to preserve the meaning of pure rotation (without dilations).

### Additional resources

3Blue1Brown's
  [Three-dimensional linear transformations](https://www.youtube.com/watch?v=rHLEWRxRGiM) shows
  visually why the columns are where the axes land. The derivation of the three matrices is in
  the Duckietown [Representations - Part 1](https://hubs.ly/Q02bk6-V0) slides and the
  [associated lecture video](https://vimeo.com/588472832?fl=pl&fe=sh), and Chapter 3 of the free
  textbook [Modern Robotics](https://hades.mech.northwestern.edu/index.php/Modern_Robotics)
  covers rotation matrices in full.

## Gimbal lock

<figure style="width:250px; margin:1.5em auto; text-align:center;">
  <img src="../assets/_images/3d_unit_sphere.jpg" style="width:100%;" />
  <figcaption style="font-size:0.9em; margin-top:0.6em; text-align:center;">
    Two angles name every point on a sphere: the longitude <i>&theta;</i> (green) and the
    latitude <i>&phi;</i> (orange). At the poles, <i>&theta;</i> is undefined.
  </figcaption>
</figure>

Consider a point on a sphere; two angles are sufficient to identify it: longitude and latitude,
  as on a globe. The trouble is at the north pole, where the latitude is 90 deg and every
  longitude gives the same point. Someone walking straight over the pole never changes
  direction, yet their longitude jumps by 180 deg the moment they cross. Nothing is wrong with
  the sphere or the person's movement; it is the mathematical notation describing the angles
  that somehow breaks.

Euler angles break in the same way, when pitch reaches $\beta = \pm 90$ deg. At $\beta = -90$
  deg the airplane points straight up. Roll spins it about its nose, which now points up, and
  yaw spins it about the vertical axis. Roll and yaw now do the same thing, so only their sum
  $\alpha + \gamma$ matters.

<figure style="width:760px; margin:1.5em auto; text-align:center;">
  <img src="../assets/_images/airplane_pitch_0.jpg" style="width:48%; display:inline-block;" />
  <img src="../assets/_images/airplane_pitch_90.jpg" style="width:48%; display:inline-block;" />
  <figcaption style="font-size:0.9em; margin-top:0.6em; text-align:center;">
    Left: the airplane level, nose along <i>x</i>. Right: pitched to
    <i>&beta;</i> = &minus;90 deg, nose pointing up. Roll and yaw now turn it about the
    same vertical line, so the blue wing tip can only move around a circle.
  </figcaption>
</figure>

With two angles doing the same job, one kind of motion is left without an angle to produce it.
  Pitch still tips the nose toward $x$, but no small change of the three angles tips it
  sideways, toward $y$. To tilt the nose sideways by just 0.001 rad, roll and yaw each have to
  jump by 90 deg, the pole crossing again. That is gimbal lock: at one orientation, one
  direction of rotation is lost.

The name comes from the gimbal, a set of three nested rings that each turn about one axis. Turn
  the middle ring by 90 deg and the other two line up, so the device physically cannot turn
  about the third axis. The video
  [Euler (gimbal lock) Explained](https://youtu.be/zc8b2Jo7mno?t=39) animates this with a set of
  gimbal rings. The activity below shows the same effect on the airplane.

## Activity: watch gimbal lock happen

The widget below rotates a toy airplane through the matrix above. The red dot is the nose, and
  the blue dot is the wing tip on the $+y$ side, the same point as in the figure.

With pitch at zero, each slider moves the airplane in a different way. Now set pitch to -90 deg
  and, before touching the other two sliders, predict: can roll and yaw each be changed by 30
  deg in a way that leaves the airplane exactly where it is? Then try it. The readout above the
  plot reports the only combination of roll and yaw that still matters at this orientation.

Repeat at pitch +90 deg, where the nose points down. The readout changes, and so does the answer
  to the prediction.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mpl_toolkits.mplot3d  # noqa: F401  (registers the 3d projection)
from ipywidgets import interact, FloatSlider

# Symbols follow the notebook:
#   roll, pitch, yaw   the angles alpha, beta, gamma, in degrees for the sliders
#   a, b, g            the same three angles in radians, as used in the matrices

# a toy airplane: nose first, then fuselage, tail fin and the two wings
airplane = np.array((
    np.concatenate((1.15, np.linspace(1, -.75, 25), np.linspace(-.75, -1, 15),
                    np.linspace(.4, 0, 10), np.linspace(.4, 0, 10)), axis=None),
    np.concatenate((0, np.linspace(0, 0, 25), np.linspace(0, 0, 15),
                    np.linspace(0, .9, 10), np.linspace(0, -.9, 10)), axis=None),
    np.concatenate((0, np.linspace(0, 0, 25), np.linspace(0, .5, 15),
                    np.linspace(0, 0, 10), np.linspace(0, 0, 10)), axis=None),
))
WING_TIP = np.argmax(airplane[1])  # the tip of the wing on the +y side


def rotation_matrix(roll_deg, pitch_deg, yaw_deg):
    """Build R = RZ @ RY @ RX, the ZYX convention from the section above."""
    a, b, g = np.deg2rad([roll_deg, pitch_deg, yaw_deg])
    rx = np.array([[1, 0, 0],
                   [0, np.cos(a), -np.sin(a)],
                   [0, np.sin(a), np.cos(a)]])
    ry = np.array([[np.cos(b), 0, np.sin(b)],
                   [0, 1, 0],
                   [-np.sin(b), 0, np.cos(b)]])
    rz = np.array([[np.cos(g), -np.sin(g), 0],
                   [np.sin(g), np.cos(g), 0],
                   [0, 0, 1]])
    return rz @ ry @ rx


def show(roll=0.0, pitch=0.0, yaw=0.0):
    pts = rotation_matrix(roll, pitch, yaw) @ airplane

    fig = plt.figure(figsize=(6.5, 6))
    ax = fig.add_subplot(projection='3d')
    ax.view_init(elev=10, azim=30)

    for vec in [(1.5, 0, 0), (0, 1.5, 0), (0, 0, 1.5)]:
        ax.quiver(0, 0, 0, *vec, color='0.6', arrow_length_ratio=0.06)
    ax.text(1.3, 0, 0, 'x', color='0.4')
    ax.text(0, 1.1, 0, 'y', color='0.4')
    ax.text(0, 0, 1.1, 'z', color='0.4')

    ax.scatter(pts[0, 1:], pts[1, 1:], pts[2, 1:], c='black', s=60)
    ax.scatter(pts[0, 0], pts[1, 0], pts[2, 0], c='red', s=120)
    ax.scatter(*pts[:, WING_TIP], c='tab:blue', s=120)

    for lim in (ax.set_xlim3d, ax.set_ylim3d, ax.set_zlim3d):
        lim(-1.2, 1.2)

    if abs(abs(pitch) - 90.0) < 1.0:
        # nose up: the two angles add; nose down: they subtract
        op, locked = ('+', roll + yaw) if pitch < 0 else ('-', roll - yaw)
        ax.set_title(f'gimbal lock: roll and yaw act on the same axis\n'
                     f'only roll {op} yaw matters, and it is {locked:+.0f} deg',
                     color='tab:red')
    else:
        ax.set_title(f'roll {roll:+.0f} deg   pitch {pitch:+.0f} deg   yaw {yaw:+.0f} deg')

    fig.tight_layout()
    plt.show()


interact(
    show,
    roll=FloatSlider(value=0.0, min=-180, max=180, step=1,
                     description='roll', continuous_update=False),
    pitch=FloatSlider(value=0.0, min=-90, max=90, step=1,
                      description='pitch', continuous_update=False),
    yaw=FloatSlider(value=0.0, min=-180, max=180, step=1,
                    description='yaw', continuous_update=False),
);

## Quaternions: the practical way out

A hovering Duckiedrone never gets close to 90 deg of pitch, so why does gimbal lock matter? The
  gyroscope reports how fast the Duckiedrone turns, never an angle, so orientation is found by
  adding up those turn rates over time.

If the orientation is kept as Euler angles, every reading first has to be converted into a
  change of roll, pitch and yaw, and that conversion divides by the cosine of the pitch. Near 90
  deg the cosine is close to zero and the result blows up: this is a mathematical singularity.
  In practice the trouble starts even earlier: at 80 deg of pitch, gyroscope noise is already
  amplified almost six times on its way into yaw.

Changing the order of the three angles only moves the bad spot somewhere else. Any description
  with exactly three numbers has one, like the pole on the sphere. The rotation matrix has no
  bad spot, but it needs nine numbers that must be kept consistent with each other.

A quaternion has no bad spot either, and needs only four numbers. Using more than three numbers
  is what avoids the singularity.

### What a quaternion is

Euler angles decompose any orientation into three subsequent rotations. Euler also showed that
  any orientation can be reached from any other with a single turn, by some angle $\psi$ about
  an axis $\mathbf{u}$ oriented in space
  ([Euler's rotation theorem](https://en.wikipedia.org/wiki/Euler%27s_rotation_theorem)). A
  quaternion stores this information as four scalars, $q = (q_x, q_y, q_z, q_w)$:

<figure style="width:420px; margin:1.5em auto; text-align:center;">
  <img src="../assets/_images/quaternion_axis_angle.png" style="width:100%;" />
  <figcaption style="font-size:0.9em; margin-top:0.6em; text-align:center;">
    The axis <b>u</b> (blue) and the angle <i>&psi;</i> (red) that a quaternion stores.
  </figcaption>
</figure>

$q_x$, $q_y$ and $q_z$ give the axis of the turn, and $q_w$ how far to turn. The exact recipe
  uses half the angle:

$$
q = \Big(u_x \sin\tfrac{\psi}{2},\; u_y \sin\tfrac{\psi}{2},\; u_z \sin\tfrac{\psi}{2},\;
  \cos\tfrac{\psi}{2}\Big).
$$

Two examples:

- no turn at all, $\psi = 0$: $q = (0, 0, 0, 1)$

- a quarter turn of yaw, $\psi = 90$ deg about $\mathbf{u} = (0, 0, 1)$:
  $q = (0, 0, \sin 45^{\circ}, \cos 45^{\circ}) = (0, 0, 0.707, 0.707)$.

Why half the angle is a longer story, and this video, for example, tells it well:
  [Quaternions and 3d rotation, explained interactively](https://www.youtube.com/watch?v=zjMuIxRvygQ).
  For this notebook the definition is enough to read a quaternion.

As for rotation matrices, successive turns can be expressed by multiplying their quaternions,
  with a multiplication rule of their own. The full algebra of quaternions is beyond the scope
  of this notebook, and many resources cover it, for example:
  [Visualizing quaternions](https://eater.net/quaternions) by Ben Eater and 3Blue1Brown builds
  it up interactively.

On a less mathematical note: some implementations, such as ROS 2, list $q_w$ last, as this
  notebook does, but other libraries list it first, so check the order whenever a quaternion
  passes from one library to another. Mixing up the two conventions might result in bad crashes.

### Three properties of quaternions

- **Unit length.** Only quaternions of unit length describe a rotation, so:
  $\sqrt{q_x^{2} + q_y^{2} + q_z^{2} + q_w^{2}} = 1$.

- **No gimbal lock.** A small turn always changes the four numbers only a little, whatever the
  orientation, so there is no singularity like pitch = 90 deg for Euler angles.

- **Two quaternions for every orientation.** $q$ and $-q$ describe the same orientation, so two
  quaternions can look opposite and still mean the same thing. Compare orientations, not
  numbers.

### Converting to Euler angles

Quaternions have a downside as well: they are hard (for a human) to read and interpret
  meaningfully at a glance; while roll, pitch and yaw are less troublesome. So quaternions are
  converted to Euler angles whenever a person or a log file needs to read an orientation. For
  the ZYX convention, the following conversion is used, with $q = (q_x, q_y, q_z, q_w)$:

$$
\begin{aligned}
\alpha \;(\text{roll}) &= \operatorname{atan2}\!\big(2(q_wq_x + q_yq_z),\; 1 - 2(q_x^{2} + q_y^{2})\big), \\
\beta \;(\text{pitch}) &= \arcsin\!\big(2(q_wq_y - q_zq_x)\big), \\
\gamma \;(\text{yaw}) &= \operatorname{atan2}\!\big(2(q_wq_z + q_xq_y),\; 1 - 2(q_y^{2} + q_z^{2})\big).
\end{aligned}
$$

The helper `quaternion_to_euler` in `imu_math.py`, which
  [Notebook 4](./4-interfacing-with-imu.ipynb) uses, implements this.

Gimbal lock comes back here. Pitch uses $\arcsin$, which only covers -90 to +90 deg. At pitch
  $\pm 90$ deg both inputs of each $\operatorname{atan2}$ become zero, so roll and yaw can no
  longer be told apart. This is why the flight controller keeps orientation as a quaternion the
  whole time and converts only when a person needs to read it. It is also why the ROS 2
  `sensor_msgs/Imu` message reports orientation as a quaternion rather than as roll, pitch and
  yaw. ROS 2 names the four numbers `x`, `y`, `z` and `w`:

```
geometry_msgs/Quaternion orientation
    float64 x
    float64 y
    float64 z
    float64 w
```

To see quaternions in ROS 2 code, the tutorial
  [Quaternion fundamentals](https://docs.ros.org/en/jazzy/Tutorials/Intermediate/Tf2/Quaternion-Fundamentals.html)
  shows how they are created and combined.

## Summary and next steps

The orientation of the Duckiedrone is the rotation between its body frame and the world frame,
  and there are three common ways to write it down:

| | Numbers | Easy to read | Bad spot | Best for |
|---|---|---|---|---|
| Euler angles | 3 | yes | gimbal lock at pitch = +/-90 deg | showing orientation to a person |
| Rotation matrix | 9 | no | none | turning a body-frame vector into a world-frame one |
| Quaternion | 4 | no | none, but $q$ and $-q$ are the same orientation | storing, estimating and sending orientation |

Any description with only three numbers has a bad spot, which is why the flight controller works
  with quaternions and conversion to roll, pitch and yaw is left for display.

The flight controller's orientation estimate reaches ROS 2 as a quaternion, in the `orientation`
  field of the IMU message. [Notebook 3](./3-duckiedrone-imu-architecture.ipynb) follows that
  value, and the measurements beside it, from the chip on the flight controller to the topic a
  program subscribes to.